# 基本設定

In [30]:
# 可調參數
EXCEL_PATH = "/Users/wanghao/8k/8K_Filings.xlsx"
SHEET_INDEX = 0

HTML_DIR = "/Volumes/One Touch/8k_file_html"
TXT_DIR  = "/Volumes/One Touch/8k_file_txt"
OTHER_DIR = "/Volumes/One Touch/8k_file_neither"

DRY_RUN = True   # True: 先跑少量測試
DRY_RUN_N = 250

MIN_INTERVAL = 0.25
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome Safari (Contact: haowang5277@gmail.com)"
)
SEC_PREFIX = "https://www.sec.gov/"


# 匯入套件

In [14]:
import re
import os
import time
from typing import List, Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs


# 讀 Excel 與去重複函式

In [16]:
start = time.perf_counter()

KEEP = ["coname", "cik", "accession"]

def keep_and_dedupe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    return out[KEEP].drop_duplicates(subset=["cik", "accession"], keep="first")

df_raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_INDEX)
res = keep_and_dedupe(df_raw)

print(f"刪減後（列, 欄）= {res.shape}")
print(res.head())

end = time.perf_counter()
print(f"花費時間：{end - start:.2f} 秒")


刪減後（列, 欄）= (835233, 3)
                               coname     cik             accession
0                   SOUTHERN UNION CO  203248  0000913907-94-000002
1  CLEVELAND ELECTRIC ILLUMINATING CO   20947  0000774197-94-000001
2                    TOLEDO EDISON CO  352049  0000774197-94-000001
3                    BROOKE GROUP LTD   59440  0000950123-94-000010
4                    MDC HOLDINGS INC  773141  0000950109-94-000014
花費時間：36.93 秒


# HTTP 與速率控制

In [18]:
class FatalError(RuntimeError):
    pass

_last_call_ts = 0.0

def _rate_limit(): #控制下載頻率，確保請求間隔不低於 MIN_INTERVAL 秒
    global _last_call_ts
    now = time.monotonic()
    wait = (_last_call_ts + MIN_INTERVAL) - now
    if wait > 0:
        time.sleep(wait)
    _last_call_ts = time.monotonic()

def _get(url: str, stream: bool = False) -> requests.Response: #對單一 URL 發送 GET 請求，內建重試、錯誤處理和限速
    _rate_limit()
    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30, stream=stream)
        if r.status_code == 429:
            ra = r.headers.get("Retry-After")
            sleep_s = int(ra) if ra and ra.isdigit() else 5
            print(f"[429] Too Many Requests: 等待 {sleep_s} 秒後重試...")
            time.sleep(sleep_s)
            _rate_limit()
            r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30, stream=stream)
        if r.status_code != 200:
            raise FatalError(f"HTTP {r.status_code}: {url}")
        return r
    except requests.RequestException as e:
        raise FatalError(f"請求失敗: {url}: {e}")


# 找到要的下載連結函式

In [26]:
_HTML_EXTS = (".htm", ".html")
_TXT_EXTS = (".txt",)

def _find_primary_doc_url(index_html: str) -> str:
    soup = BeautifulSoup(index_html, "html.parser")

    # 找 Document Format Files 表格
    table = None
    for t in soup.find_all("table"):
        ths = [th.get_text(strip=True).lower() for th in t.find_all("th")]
        if {"seq", "description", "document", "type", "size"} <= set(ths):
            table = t
            break
    if table is None:
        raise FatalError("找不到 Document Format Files 表格")

    ths = [th.get_text(strip=True).lower() for th in table.find_all("th")]
    col = {name: ths.index(name) for name in ["seq", "description", "document", "type", "size"]}

    # 目標表單：8-K 家族
    keywords = ("8-k", "8-k/a")

    candidates_html = []
    candidates_txt = []

    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) <= max(col.values()):
            continue

        type_txt = tds[col["type"]].get_text(strip=True).lower()

        # 等於或前綴（更準確，避免誤抓）
        if not any(type_txt == k or type_txt.startswith(k) for k in keywords):
            continue

        for a in tds[col["document"]].find_all("a", href=True):
            raw = a["href"].strip()
            low = raw.lower()

            # 處理 iXBRL 連結：/ix?doc=...
            if low.startswith("/ix?") or "/ix?" in low:
                q = urlparse(raw).query
                doc = parse_qs(q).get("doc", [None])[0]
                if doc:
                    raw = doc
                    low = raw.lower()

            # 避免 viewer 殼連結
            if "viewer" in low and ("doc=" not in low) and not low.endswith(_HTML_EXTS + _TXT_EXTS):
                continue

            # 只接受 .htm/.html/.txt
            if not low.endswith(_HTML_EXTS + _TXT_EXTS):
                continue

            abs_url = urljoin(SEC_PREFIX, raw)
            if low.endswith(_HTML_EXTS):
                candidates_html.append(abs_url)
            else:
                candidates_txt.append(abs_url)

    # 優先回傳 HTML，再回傳 TXT
    if candidates_html:
        return candidates_html[0]
    if candidates_txt:
        return candidates_txt[0]

    # --- Fallback：沒匹配到 Type，挑第一個 HTML/TXT ---
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) <= max(col.values()):
            continue
        for a in tds[col["document"]].find_all("a", href=True):
            raw = a["href"].strip()
            low = raw.lower()

            if low.startswith("/ix?") or "/ix?" in low:
                q = urlparse(raw).query
                doc = parse_qs(q).get("doc", [None])[0]
                if doc:
                    raw = doc
                    low = raw.lower()

            if low.endswith(_HTML_EXTS):
                return urljoin(SEC_PREFIX, raw)
            if low.endswith(_TXT_EXTS):
                candidates_txt.append(urljoin(SEC_PREFIX, raw))

    if candidates_txt:
        return candidates_txt[0]

    raise FatalError("找不到符合 8-K/8-KA 的主檔連結，且沒有可用的 fallback")


# 組出 Index URL + 下載函式

In [22]:
def _sanitize_filename(name: str) -> str:
    """移除檔名中不合法的字元並用底線取代空白"""
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = name.replace(" ", "_")
    return name.strip()

def _make_index_url(cik: str, accession: str) -> str:
    acc_no_dash = accession.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_no_dash}/{accession}-index.html"

def _ensure_dirs(*dirs):
    for d in dirs:
        os.makedirs(d, exist_ok=True)

def download_10k(df: pd.DataFrame, limit: int = 100) -> List[str]:
    #_ensure_dirs(HTML_DIR, TXT_DIR, OTHER_DIR)
    urls = [_make_index_url(row.cik, row.accession) for _, row in df.iterrows()]
    urls = urls[:limit]

    saved = []
    errors = []

    start_t = time.time()
    for i, (_, row) in enumerate(df.iloc[:limit].iterrows(), 1):
        coname_safe = _sanitize_filename(row.coname)
        accession_safe = _sanitize_filename(row.accession)
        url = _make_index_url(row.cik, row.accession)

        try:
            index_resp = _get(url)
            primary_url = _find_primary_doc_url(index_resp.text)
            resp = _get(primary_url, stream=True)

            # 檔名格式：公司名_accession.html
            filename = f"{coname_safe}_{accession_safe}"
            if primary_url.lower().endswith(_HTML_EXTS):
                path = os.path.join(HTML_DIR, f"{filename}.html")
            elif primary_url.lower().endswith(_TXT_EXTS):
                path = os.path.join(TXT_DIR, f"{filename}.txt")
            else:
                path = os.path.join(OTHER_DIR, f"{filename}.dat")

            # 寫入檔案
            with open(path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024 * 64):
                    if chunk:
                        f.write(chunk)

            saved.append(path)
            print(f"[OK] {i}/{len(urls)} -> {path}")

        except Exception as e:
            errors.append({
                "coname": row.coname,
                "accession": row.accession,
                "index_url": url,
                "error": str(e)
            })
            print(f"[ERR] {i}/{len(urls)}: {e}")

    print(f"\n✅ 成功下載 {len(saved)} 檔，失敗 {len(errors)} 檔；耗時 {time.time() - start_t:.2f} 秒")

    if errors:
        errors_df = pd.DataFrame(errors)
        errors_df.to_csv("download_errors.csv", index=False, encoding="utf-8-sig")
        print(f"❗ 已記錄失敗清單到 download_errors.csv ({len(errors)} 筆)")

    return saved


# 執行下載

In [32]:
if DRY_RUN:
    df_slice = res.iloc[:DRY_RUN_N]
else:
    df_slice = res

print(f"本次處理筆數：{len(df_slice)}")
saved_files = download_10k(df_slice, limit=len(df_slice))


本次處理筆數：250
[OK] 1/250 -> /Volumes/One Touch/8k_file_txt/SOUTHERN_UNION_CO_0000913907-94-000002.txt
[OK] 2/250 -> /Volumes/One Touch/8k_file_txt/CLEVELAND_ELECTRIC_ILLUMINATING_CO_0000774197-94-000001.txt
[OK] 3/250 -> /Volumes/One Touch/8k_file_txt/TOLEDO_EDISON_CO_0000774197-94-000001.txt
[OK] 4/250 -> /Volumes/One Touch/8k_file_txt/BROOKE_GROUP_LTD_0000950123-94-000010.txt
[OK] 5/250 -> /Volumes/One Touch/8k_file_txt/MDC_HOLDINGS_INC_0000950109-94-000014.txt
[OK] 6/250 -> /Volumes/One Touch/8k_file_txt/CENTRAL_HUDSON_GAS_&_ELECTRIC_CORP_0000018647-94-000001.txt
[OK] 7/250 -> /Volumes/One Touch/8k_file_txt/CITIZENS_UTILITIES_CO_0000020520-94-000001.txt
[OK] 8/250 -> /Volumes/One Touch/8k_file_txt/PRICE_COMMUNICATIONS_CORP_0000897446-94-000004.txt
[OK] 9/250 -> /Volumes/One Touch/8k_file_txt/KAYDON_CORP_0000740694-94-000001.txt
[OK] 10/250 -> /Volumes/One Touch/8k_file_txt/PENTAIR_INC_0000077360-94-000007.txt
[OK] 11/250 -> /Volumes/One Touch/8k_file_txt/SILGAN_HOLDINGS_INC_0000896058-